## Upload Vectors to PostgreSQL Database

### Installing Utilities and Libraries

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 python-dotenv openai==2.38.0 langchain-community==0.4.1

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()

# Loading the database configurations
host = os.getenv("DATABASE_HOSTNAME")
db_name = os.getenv("DATABASE_NAME")
username = os.getenv("DATABASE_USERNAME")
password = os.getenv("DATABASE_PASSWORD")

# Loading the Azure OpenAI configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")

### Create the connection pool 

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    azure_endpoint=azure_openai_endpoint,
    api_version="2024-06-01"
)

### Create the Document Chunker Helper Function with LangChain

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def perform_fixed_size_chunking(document, chunk_size=500, chunk_overlap=50):
    """
    Performs recursive chunking on a document with specified overlap.
    Uses RecursiveCharacterTextSplitter which tries multiple separators.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return text_splitter.split_text(document)

### Create the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(text):

    response = azure_openai_client.embeddings.create(
        model=embedding_model_name,
        input=text
    )

    return response.data[0].embedding

### Fetch the Report Data from Table

In [ ]:
fetch_query = """
SELECT
    RecordID,
    CompanyName,
    SustainabilityReport
FROM RAG.ESG_TextData
"""

In [ ]:
from psycopg.rows import dict_row

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(fetch_query)

        esg_documents = cur.fetchall()

### Generate Chunks

In [ ]:
all_chunks = []

for document in esg_documents:

    chunks = perform_fixed_size_chunking(
        document["sustainabilityreport"]
    )

    for chunk in chunks:

        all_chunks.append(
            {
                "record_id":
                    document["recordid"],

                "company_name":
                    document["companyname"],

                "chunk_text":
                    chunk
            }
        )

### Generate Embeddings

In [ ]:
for chunk in all_chunks:

    chunk["embedding"] = generate_embeddings(
        chunk["chunk_text"]
    )

### Insert Chunks into Table

In [ ]:
insert_query = """
INSERT INTO RAG.ESG_Chunks
(
    RecordID,
    CompanyName,
    ChunkText,
    ChunkEmbedding
)
VALUES
(
    %s,
    %s,
    %s,
    %s
)
"""

In [ ]:
with pool.connection() as conn:

    with conn.cursor() as cur:

        for chunk in all_chunks:

            cur.execute(
                insert_query,
                (
                    chunk["record_id"],
                    chunk["company_name"],
                    chunk["chunk_text"],
                    chunk["embedding"]
                )
            )

        conn.commit()

### Create the HNSW Vector Index

In [ ]:
create_index_query = """
CREATE INDEX IF NOT EXISTS esg_chunks_embedding_hnsw_idx
ON RAG.ESG_Chunks
USING hnsw
(
    ChunkEmbedding vector_cosine_ops
)
WITH
(
    m = 16,
    ef_construction = 64
);
"""

with pool.connection() as conn:

    with conn.cursor() as cur:

        cur.execute(create_index_query)

    conn.commit()

print("HNSW Index Created Successfully")

### Verify the Index

In [ ]:
verify_query = """
SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE schemaname = 'rag'
"""

In [ ]:
from psycopg.rows import dict_row

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(verify_query)

        indexes = cur.fetchall()

for index in indexes:

    print(index["indexname"])
    print(index["indexdef"])
    print("---------------")